# MaaS Rate Limiting Demo

This notebook demonstrates RHOAI 3.4 MaaS governance features:
1. **API Key authentication** — using `sk-oai-` keys generated from Gen AI Studio
2. **Token rate limiting** — MaaSSubscription enforces token budgets per subscription
3. **429 behavior** — what happens when you exceed your token limit
4. **Burst vs sustained load** — how rate limiting behaves under different patterns

## Prerequisites
- A deployed model published to MaaS (`MaaSModelRef` + `MaaSSubscription` + `MaaSAuthPolicy`)
- An API key generated from **Gen AI Studio > API Keys**
- The MaaS rate limiting fixes applied (see `docs/maas-token-ratelimit-span-buffer-bug.md`)

## Setup

In [ ]:
import os
import time
import json
import requests
import warnings
from datetime import datetime
from collections import defaultdict

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not installed -- charts will be skipped")
    print("Install with: !pip install matplotlib")

## Configuration

Set your MaaS API key and endpoint. Get the API key from:
**RHOAI Dashboard → Gen AI Studio → API Keys → Create API key**

In [ ]:
import subprocess

API_KEY = os.environ.get("MAAS_API_KEY", "sk-oai-PASTE_YOUR_KEY_HERE")

MAAS_ENDPOINT = "https://maas.apps.cluster-9tjvr.9tjvr.sandbox2001.opentlc.com"
MODEL_NAMESPACE = "0-test"
MODEL_NAME = "redhataiqwen3-8b-fp8-dynamic"

CHAT_URL = f"{MAAS_ENDPOINT}/{MODEL_NAMESPACE}/{MODEL_NAME}/v1/chat/completions"

print(f"Endpoint:  {CHAT_URL}")
print(f"Model:     {MODEL_NAMESPACE}/{MODEL_NAME}")
print(f"API Key:   {API_KEY[:15]}..." if len(API_KEY) > 15 else f"API Key: {API_KEY}")
if API_KEY.startswith("sk-oai-PASTE"):
    print("\n** Paste your API key above. Get one from: RHOAI Dashboard > Gen AI Studio > API Keys **")

## Helper Functions

In [ ]:
def chat_request(prompt, max_tokens=100):
    """Send a chat completion request and return response + metadata."""
    start = time.time()
    try:
        resp = requests.post(
            CHAT_URL,
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens,
            },
            verify=False,
            timeout=60,
        )
        elapsed = time.time() - start
        result = {
            "status": resp.status_code,
            "elapsed_s": round(elapsed, 2),
            "timestamp": datetime.now(),
        }

        if resp.status_code == 200:
            data = resp.json()
            usage = data.get("usage", {})
            result["prompt_tokens"] = usage.get("prompt_tokens", 0)
            result["completion_tokens"] = usage.get("completion_tokens", 0)
            result["total_tokens"] = usage.get("total_tokens", 0)
            result["response"] = data["choices"][0]["message"]["content"][:100]
        elif resp.status_code == 429:
            result["error"] = "RATE LIMITED"
            retry_after = resp.headers.get("Retry-After", "unknown")
            result["retry_after"] = retry_after
        else:
            result["error"] = resp.text[:200]

        return result
    except Exception as e:
        return {
            "status": 0,
            "elapsed_s": round(time.time() - start, 2),
            "timestamp": datetime.now(),
            "error": str(e),
        }


def print_result(i, result):
    status = result['status']
    elapsed = result['elapsed_s']

    if status == 200:
        tokens = result.get('total_tokens', '?')
        print(f"  [{i:3d}] ✅ 200 | {tokens:5} tokens | {elapsed:.1f}s")
    elif status == 429:
        retry = result.get('retry_after', '?')
        print(f"  [{i:3d}] 🚫 429 RATE LIMITED | retry-after: {retry}s | {elapsed:.1f}s")
    else:
        err = result.get('error', 'unknown')[:60]
        print(f"  [{i:3d}] ❌ {status} | {err} | {elapsed:.1f}s")

## Test 1: Verify API Key Works

Send a single request to confirm the API key, endpoint, and model are all working.

In [ ]:
print("Sending test request...")
result = chat_request("Say hello in exactly 5 words.", max_tokens=20)

if result["status"] == 200:
    print(f"\n✅ API key works!")
    print(f"   Model: {MODEL_NAME}")
    print(f"   Response: {result['response']}")
    print(f"   Tokens used: {result['total_tokens']} (prompt: {result['prompt_tokens']}, completion: {result['completion_tokens']})")
    print(f"   Latency: {result['elapsed_s']}s")
else:
    print(f"\n❌ Request failed: {result}")
    print("\nTroubleshooting:")
    print("  - Is the API key correct? (should start with sk-oai-)")
    print("  - Is the endpoint URL correct?")
    print("  - Is the model deployed and ready?")
    print(f"  - Try: curl -sk {CHAT_URL} -H 'Authorization: Bearer {API_KEY[:10]}...'")

## Test 2: Check Current Rate Limit Status

The MaaS gateway returns rate limit headers showing your remaining token budget.

In [ ]:
resp = requests.post(
    CHAT_URL,
    headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    json={"model": MODEL_NAME, "messages": [{"role": "user", "content": "hi"}], "max_tokens": 5},
    verify=False, timeout=30,
)

print("Rate limit headers:")
for header, value in sorted(resp.headers.items()):
    if any(k in header.lower() for k in ['ratelimit', 'rate-limit', 'retry', 'x-ratelimit', 'remaining', 'limit', 'reset']):
        print(f"  {header}: {value}")

if not any('ratelimit' in h.lower() or 'rate-limit' in h.lower() for h in resp.headers):
    print("  (no rate limit headers found -- rate limiting may not be configured)")
    print("  Check: oc get tokenratelimitpolicy -n models-as-a-service")

## Test 3: Sustained Load — Exhaust Token Budget

Send requests in a loop until we hit the rate limit (HTTP 429).
Each request generates ~100 tokens. Adjust `MAX_REQUESTS` based on your subscription's token limit.

**Example:** If your `MaaSSubscription` has a 10,000 tokens/hour limit, ~100 requests × 100 tokens = 10,000 tokens should trigger the limit.

In [ ]:
MAX_REQUESTS = 50
TOKENS_PER_REQUEST = 100
DELAY_BETWEEN_REQUESTS = 0.5  # seconds

results = []
total_tokens_used = 0
rate_limited_at = None

print(f"Sending {MAX_REQUESTS} requests ({TOKENS_PER_REQUEST} max tokens each)...")
print(f"Delay between requests: {DELAY_BETWEEN_REQUESTS}s")
print("="*70)

for i in range(1, MAX_REQUESTS + 1):
    result = chat_request(
        f"Count from 1 to 50. Request number {i}.",
        max_tokens=TOKENS_PER_REQUEST,
    )
    results.append(result)
    print_result(i, result)

    if result["status"] == 200:
        total_tokens_used += result.get("total_tokens", 0)
    elif result["status"] == 429:
        if rate_limited_at is None:
            rate_limited_at = i
        print(f"\n🚫 Rate limit hit after {total_tokens_used} total tokens ({i} requests)")
        break

    time.sleep(DELAY_BETWEEN_REQUESTS)

print("\n" + "="*70)
print(f"Total requests: {len(results)}")
print(f"Total tokens consumed: {total_tokens_used}")
if rate_limited_at:
    print(f"Rate limited at request #{rate_limited_at}")
else:
    print("Rate limit NOT reached (increase MAX_REQUESTS or lower subscription limit)")

## Test 4: Burst Load — Rapid Fire Requests

Send requests as fast as possible (no delay) to test how the gateway handles bursts.
This simulates a misbehaving client or DoS scenario.

In [ ]:
BURST_COUNT = 20

burst_results = []
print(f"Sending {BURST_COUNT} requests with NO delay (burst mode)...")
print("="*70)

for i in range(1, BURST_COUNT + 1):
    result = chat_request("Say OK.", max_tokens=5)
    burst_results.append(result)
    print_result(i, result)

statuses = defaultdict(int)
for r in burst_results:
    statuses[r["status"]] += 1

print("\n" + "="*70)
print("Burst results summary:")
for status, count in sorted(statuses.items()):
    label = {200: "OK", 429: "RATE LIMITED", 503: "SERVICE UNAVAILABLE"}.get(status, "OTHER")
    pct = count / len(burst_results) * 100
    print(f"  HTTP {status} ({label}): {count}/{len(burst_results)} ({pct:.0f}%)")

## Test 5: Visualize Token Consumption vs Rate Limit

Chart the cumulative token usage over time with the rate limit threshold.

In [ ]:
if not HAS_MATPLOTLIB:
    print("Skipping chart -- install matplotlib: !pip install matplotlib")
elif not results:
    print("No results from Test 3 -- run that cell first")
else:
    successful = [r for r in results if r["status"] == 200]
    if not successful:
        print("No successful requests to chart")
    else:
        timestamps = [r["timestamp"] for r in successful]
        cumulative_tokens = []
        running_total = 0
        for r in successful:
            running_total += r.get("total_tokens", 0)
            cumulative_tokens.append(running_total)

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        # Cumulative token usage
        ax1.plot(timestamps, cumulative_tokens, 'b-o', markersize=4, label='Cumulative tokens')
        ax1.set_ylabel('Cumulative Tokens')
        ax1.set_title('MaaS Token Consumption Over Time')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Mark rate limit hit
        rate_limited_results = [r for r in results if r["status"] == 429]
        if rate_limited_results:
            rl_time = rate_limited_results[0]["timestamp"]
            ax1.axvline(x=rl_time, color='red', linestyle='--', label=f'Rate limit hit ({running_total} tokens)')
            ax1.legend()

        # Per-request tokens
        per_request = [r.get("total_tokens", 0) for r in successful]
        ax2.bar(range(len(per_request)), per_request, color='steelblue', alpha=0.7)
        ax2.set_xlabel('Request #')
        ax2.set_ylabel('Tokens per Request')
        ax2.set_title('Tokens Per Request')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('maas-ratelimit-results.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("\nChart saved to maas-ratelimit-results.png")

## Test 6: Compare Two API Keys (Different Subscriptions)

If you have two API keys from different subscriptions (e.g. premium vs free tier),
you can compare their rate limits side by side.

In [ ]:
KEY_A = API_KEY  # Your primary key (already set above)
KEY_B = "sk-oai-PASTE_SECOND_KEY_HERE"  # A second key with different limits

SKIP_COMPARISON = KEY_B.startswith("sk-oai-PASTE")

if SKIP_COMPARISON:
    print("Set KEY_B above to a second API key to run this comparison.")
    print("Create two subscriptions with different token limits:")
    print("  - Subscription A: 50,000 tokens/hour (premium)")
    print("  - Subscription B: 5,000 tokens/hour (free)")
else:
    for label, key in [("Key A", KEY_A), ("Key B", KEY_B)]:
        original_key = API_KEY
        API_KEY = key
        print(f"\n{'='*40}")
        print(f"Testing {label}: {key[:15]}...")
        print(f"{'='*40}")

        tokens_used = 0
        for i in range(1, 11):
            r = chat_request("Count to 20.", max_tokens=50)
            print_result(i, r)
            if r["status"] == 200:
                tokens_used += r.get("total_tokens", 0)
            elif r["status"] == 429:
                break
            time.sleep(0.3)

        print(f"  → {label} used {tokens_used} tokens")
        API_KEY = original_key

## Understanding the Results

### What the MaaSSubscription controls

```yaml
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSSubscription
metadata:
  name: demo-sub
  namespace: models-as-a-service
spec:
  modelRefs:
    - name: qwen3-32b
      namespace: admin-workshop
      tokenRateLimits:
        - limit: 10000      # <-- total tokens allowed
          window: "1h"      # <-- per hour
  owner:
    groups:
      - name: rhods-users   # <-- who can generate API keys
```

### Request flow

```
Client (this notebook)
  → Authorization: Bearer sk-oai-xxxxx
  → OpenShift Router (passthrough)
  → Gateway Envoy (TLS termination)
  → Authorino (validates API key via PostgreSQL)
  → Kuadrant WasmPlugin (pre-check: under limit?)
  → Model Pod (vLLM generates response)
  → Kuadrant WasmPlugin (post-report: add tokens to counter)
  → Limitador (stores cumulative token count per subscription)
```

### Common issues

| Symptom | Cause | Fix |
|---------|-------|-----|
| Never get 429 | Span buffer overflow dropping token reports | Apply health check + Redis fixes (see docs) |
| 429 immediately | Already over limit from previous tests | Wait for the window to reset (e.g. 1 hour) |
| 401 Unauthorized | Invalid or expired API key | Generate a new key in Gen AI Studio |
| 403 Forbidden | MaaSAuthPolicy missing or misconfigured | Check `oc get maasauthpolicy -n models-as-a-service` |